In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("/data/Projet_reconversion/Projet et cours Data Analyst/Cours Le Wagon/projet_crise_logement/données a traiter/Copie de DVF_MEL_2020_2025.csv", sep=";", parse_dates=["Date mutation"], dayfirst=True)

regroupement des biens 1 ventes = 1 ligne 

In [17]:
df.head()

,Date mutation,Valeur fonciere,Type local,Surface reelle bati,Nombre pieces principales,Commune,Code departement,Code postal,Surface terrain
0,2020-07-01,339700,Maison,213.0,9.0,RONCHIN,59,59790.0,191.0
1,2020-07-01,339700,Dépendance,0.0,0.0,RONCHIN,59,59790.0,15.0
2,2020-07-01,339700,NaN,NaN,NaN,RONCHIN,59,59790.0,48.0
3,2020-07-01,339700,NaN,NaN,NaN,RONCHIN,59,59790.0,536.0
4,2020-07-03,550000,Maison,210.0,7.0,LILLE,59,59000.0,215.0


In [21]:
df["Type local"].value_counts()

Type local
Dépendance                                  64234
Maison                                      51503
Appartement                                 48296
Local industriel. commercial ou assimilé    12550
Name: count, dtype: int64

In [27]:
df["Date mutation"] = pd.to_datetime(df["Date mutation"], dayfirst=True)
df["Valeur fonciere"] = pd.to_numeric(df["Valeur fonciere"], errors="coerce")
df = df[df["Type local"].isin(["Maison", "Appartement"])]
df_clean = (
    df.groupby(
        ["Date mutation","Code postal","Commune","Valeur fonciere"],
        as_index=False
    )
    .agg({
        "Surface reelle bati": "sum",                # somme des surfaces bâties
        "Nombre pieces principales": "max",          # prendre le max des pièces
        "Type local": "first",                       # prendre le premier type non nul
        "Surface terrain": "sum"                     # somme des terrains
    })
)

# Ajouter prix au m² (facultatif mais pratique)
df_clean["prix_m2"] = df_clean["Valeur fonciere"] / df_clean["Surface reelle bati"]

# Vérifier le résultat
df_clean



,Date mutation,Code postal,Commune,Valeur fonciere,Surface reelle bati,Nombre pieces principales,Type local,Surface terrain,prix_m2
0,2020-07-01,59000.0,LILLE,57000.0,19.0,1.0,Appartement,0.0,3000.000000
1,2020-07-01,59000.0,LILLE,110100.0,34.0,2.0,Appartement,0.0,3238.235294
2,2020-07-01,59000.0,LILLE,113000.0,43.0,2.0,Appartement,0.0,2627.906977
3,2020-07-01,59000.0,LILLE,123500.0,55.0,2.0,Appartement,0.0,2245.454545
4,2020-07-01,59000.0,LILLE,147500.0,47.0,2.0,Appartement,0.0,3138.297872
...,...,...,...,...,...,...,...,...,...
81074,2025-06-30,59800.0,LILLE,108000.0,22.0,1.0,Appartement,0.0,4909.090909
81075,2025-06-30,59830.0,BACHY,260000.0,80.0,4.0,Maison,279.0,3250.000000
81076,2025-06-30,59890.0,QUESNOY SUR DEULE,130000.0,84.0,5.0,Maison,235.0,1547.619048
81077,2025-06-30,59960.0,NEUVILLE EN FERRAIN,252000.0,130.0,5.0,Maison,326.0,1938.461538


In [28]:
df.to_csv("DVF_clean_prix_immo.csv", index=False)